In [0]:
%run ../read_params

In [0]:
%run ../utils

In [0]:
def convert_roman(roman_str):
    """
    Doc String
    """
    if not roman_str:
        return None
    try:
        return roman.fromRoman(roman_str)
    except:
        return None
    
roman_udf = udf(convert_roman, IntegerType())

In [0]:
game_versions = (
    spark.table(f'{STAGING_DATABASE_PREFIX}.game_versions')
    .withColumn("generation_name", split(col("generation.name"), '-')[1])
    .drop('generation')
    .withColumn("generation", roman_udf(col("generation_name")))
    .withColumn("versions_exploded", explode(col('versions')))
    .select(
        col("version_group_id"),
        col("id"),
        col("name"),
        col("generation_name"),
        col("generation"),
        regexp_replace(col("versions_exploded.name"), "-", "_").alias('version_name')
    )
)

game_versions.write.format('delta').mode("overwrite").option('overwriteSchema', 'true').saveAsTable(f"{BRONZE_DATABASE_PREFIX}.game_versions")

In [0]:
pokedexes_exploded = (
    spark.table(f'{STAGING_DATABASE_PREFIX}.extracted_pokedexes')
    .withColumn('pokemon_entries', explode(col('pokemon_entries')))
    .select(
        col('id'),
        col('name'),
        col('region'),
        col('is_main_series'),
        col('pokemon_entries.entry_number').alias('pokemon_pokedex_number'),
        col('pokemon_entries.pokemon_species.name').alias('pokemon_name'),
        col('pokemon_entries.pokemon_species.url').alias('pokemon_url')
    )
)

pokedexes_exploded.write.format('delta').mode("overwrite").option('overwriteSchema', 'true').saveAsTable(f"{BRONZE_DATABASE_PREFIX}.expanded_pokedexes")